# Programación funcional · Avanzado

Composición de `map` y `filter`, estadística, rendimiento y corrutinas.

## 1. Buscador de inmuebles

Dada una lista de inmuebles, escribir una función que reciba la lista y un presupuesto y devuelva los inmuebles cuyo precio no lo supere, añadiendo a cada uno su precio. Precio base: `(metros·1000 + habitaciones·5000 + garaje·15000) · (1 − antigüedad/100)`; en la zona B se multiplica por 1.5.

In [1]:
from datetime import date

inmuebles = [
    {"año": 2000, "metros": 100, "habitaciones": 3, "garaje": True, "zona": "A"},
    {"año": 2012, "metros": 60, "habitaciones": 2, "garaje": True, "zona": "B"},
    {"año": 1980, "metros": 120, "habitaciones": 4, "garaje": False, "zona": "A"},
    {"año": 2005, "metros": 75, "habitaciones": 3, "garaje": True, "zona": "B"},
    {"año": 2015, "metros": 90, "habitaciones": 2, "garaje": False, "zona": "A"},
]

def con_precio(inmueble):
    antiguedad = date.today().year - inmueble["año"]
    precio = (inmueble["metros"] * 1000 + inmueble["habitaciones"] * 5000 + inmueble["garaje"] * 15000) * (1 - antiguedad / 100)
    return {**inmueble, "precio": precio * (1.5 if inmueble["zona"] == "B" else 1)}

def busca(inmuebles, presupuesto):
    return list(filter(lambda i: i["precio"] <= presupuesto, map(con_precio, inmuebles)))

for inmueble in busca(inmuebles, 100000):
    print(inmueble)

{'año': 2000, 'metros': 100, 'habitaciones': 3, 'garaje': True, 'zona': 'A', 'precio': 96200.0}
{'año': 1980, 'metros': 120, 'habitaciones': 4, 'garaje': False, 'zona': 'A', 'precio': 75600.0}
{'año': 2015, 'metros': 90, 'habitaciones': 2, 'garaje': False, 'zona': 'A', 'precio': 89000.0}


## 2. Valores atípicos

Escribir una función que reciba una muestra y devuelva los valores atípicos: aquellos cuya puntuación típica $\frac{x-\bar{x}}{\sigma}$ es mayor que 3 o menor que −3.

In [2]:
def atipicos(muestra):
    media = sum(muestra) / len(muestra)
    sigma = (sum((x - media) ** 2 for x in muestra) / len(muestra)) ** 0.5
    return [x for x in muestra if abs((x - media) / sigma) > 3]

print(atipicos([1, 2, 3, 2, 1, 3, 2, 2, 1, 3, 2, 2, 3, 1, 2, 50]))

[50]


## 3. Benchmark con `timeit`

Encontrar dos formas de hacer la misma tarea, comprobar con `timeit` cuál es más rápida y medir también el consumo de memoria con `tracemalloc`.

In [3]:
import timeit
import tracemalloc

def con_bucle(n):
    resultado = []
    for i in range(n):
        resultado.append(i * i)
    return resultado

def con_comprension(n):
    return [i * i for i in range(n)]

def con_generador(n):
    return sum(i * i for i in range(n))

for f in (con_bucle, con_comprension):
    print(f"{f.__name__:16} {timeit.timeit(lambda: f(100_000), number=50):.3f} s")

for f in (con_comprension, con_generador):
    tracemalloc.start()
    f(100_000)
    pico = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    print(f"{f.__name__:16} pico de memoria {pico / 1024:.0f} KiB")

con_bucle        0.135 s
con_comprension  0.123 s
con_comprension  pico de memoria 3907 KiB
con_generador    pico de memoria 1 KiB


## 4. Peticiones asíncronas con límite

Simular peticiones a una API con `asyncio`, procesando como máximo `MAX_TASKS` a la vez y gestionando las excepciones de cada petición para que un fallo no arruine el resto.

In [4]:
import asyncio
import random

MAX_TASKS = 3

async def fetch(usuario):
    # simula latencia y fallos de red
    await asyncio.sleep(random.uniform(0.05, 0.2))
    if random.random() < 0.2:
        raise ConnectionError(f"fallo al pedir {usuario}")
    return random.randint(0, 1000)

async def fetch_all(usuarios):
    semaforo = asyncio.Semaphore(MAX_TASKS)

    async def limitado(usuario):
        async with semaforo:
            return await fetch(usuario)

    resultados = await asyncio.gather(*(limitado(u) for u in usuarios), return_exceptions=True)
    return dict(zip(usuarios, resultados))

# Jupyter permite await directamente en la celda
resultados = await fetch_all([f"user{i}" for i in range(8)])
for usuario, seguidores in resultados.items():
    print(usuario, seguidores)

user0 856
user1 fallo al pedir user1
user2 293
user3 75
user4 965
user5 fallo al pedir user5
user6 fallo al pedir user6
user7 fallo al pedir user7
